# Projeto (Seção 5) — solução comentada

Continua a solução comentada da seção 4 (com o gerador `make_alias`). Enunciado em [`07_projeto`](07_projeto.ipynb).

## 1. Solução

Como `make_alias` remove o `_` final, o novo campo `id_` ganha o alias `id` **sem** precisar de `Field`. Ser o primeiro campo é só uma
questão de posição na classe (a ordem de declaração define a ordem dos campos e da serialização).

In [1]:
from datetime import date
from enum import Enum
from uuid import UUID, uuid1, uuid4
from pydantic import UUID4, BaseModel, ConfigDict, Field, ValidationError, field_serializer
from pydantic.alias_generators import to_camel


class AutomobileType(Enum):
    sedan = "Sedan"
    coupe = "Coupe"
    convertible = "Convertible"
    suv = "SUV"
    truck = "Truck"


def make_alias(field_name: str) -> str:
    return to_camel(field_name).removesuffix("_")


class Automobile(BaseModel):
    model_config = ConfigDict(
        extra="forbid",
        str_strip_whitespace=True,
        validate_default=True,
        validate_assignment=True,
        alias_generator=make_alias,
    )

    id_: UUID4 | None = None                  # alias "id" gerado pelo make_alias
    manufacturer: str
    series_name: str
    type_: AutomobileType
    is_electric: bool = False
    manufactured_date: date = Field(validation_alias="completionDate")
    base_msrp_usd: float = Field(validation_alias="msrpUSD", serialization_alias="baseMSRPUSD")
    vin: str
    number_of_doors: int = Field(default=4, validation_alias="doors")
    registration_country: str | None = None
    license_plate: str | None = None

    @field_serializer("manufactured_date", when_used="json-unless-none")
    def serialize_date(self, value: date) -> str:
        return value.strftime("%Y/%m/%d")

## 2. Testes do enunciado

In [2]:
data = {
    "id": "c4e60f4a-3c7f-4da5-9b3f-07aee50b23e7",
    "manufacturer": "BMW",
    "seriesName": "M4",
    "type": "Convertible",
    "isElectric": False,
    "completionDate": "2023-01-01",
    "msrpUSD": 93_300,
    "vin": "1234567890",
    "doors": 2,
    "registrationCountry": "France",
    "licensePlate": "AAA-BBB"
}

In [3]:
from uuid import UUID

expected_serialized_by_alias = {
    'id': UUID('c4e60f4a-3c7f-4da5-9b3f-07aee50b23e7'),
    'manufacturer': 'BMW',
    'seriesName': 'M4',
    'type': AutomobileType.convertible,
    'isElectric': False,
    'manufacturedDate': date(2023, 1, 1),
    'baseMSRPUSD': 93300.0,
    'vin': '1234567890',
    'numberOfDoors': 2,
    'registrationCountry': 'France',
    'licensePlate': 'AAA-BBB'
}

In [4]:
data_no_id = {
    "manufacturer": "BMW",
    "seriesName": "M4",
    "type": "Convertible",
    "isElectric": False,
    "completionDate": "2023-01-01",
    "msrpUSD": 93_300,
    "vin": "1234567890",
    "doors": 2,
    "registrationCountry": "France",
    "licensePlate": "AAA-BBB"
}

In [5]:
expected_serialization_data_no_id_by_alias = {
    'id': None,
    'manufacturer': 'BMW',
    'seriesName': 'M4',
    'type': AutomobileType.convertible,
    'isElectric': False,
    'manufacturedDate': date(2023, 1, 1),
    'baseMSRPUSD': 93300.0,
    'vin': '1234567890',
    'numberOfDoors': 2,
    'registrationCountry': 'France',
    'licensePlate': 'AAA-BBB'
}

In [6]:
car = Automobile.model_validate(data)
assert car.model_dump(by_alias=True) == expected_serialized_by_alias
assert list(car.model_dump(by_alias=True))[0] == "id"           # primeiro campo
car_no_id = Automobile.model_validate(data_no_id)
assert car_no_id.model_dump(by_alias=True) == expected_serialization_data_no_id_by_alias
print(f"Automobile.model_fields['id_'].alias: {Automobile.model_fields['id_'].alias}")
print("testes do enunciado: ok")

Automobile.model_fields['id_'].alias: id
testes do enunciado: ok


## 3. Testes extras

In [7]:
def error_types(payload: dict) -> list[tuple]:
    try:
        Automobile.model_validate(payload)
    except ValidationError as ex:
        return [(err["loc"], err["type"]) for err in ex.errors()]
    raise AssertionError("era esperado ValidationError")

# UUID inválido e UUID de outra versão (UUID4 valida a versão)
print(f"id='não-é-uuid': {error_types({**data, 'id': 'não-é-uuid'})}")
print(f"id=uuid1(): {error_types({**data, 'id': str(uuid1())})}")

# aceita objeto UUID e string; serializa como string no JSON
assert Automobile.model_validate({**data, "id": uuid4()}).id_.version == 4
import json
print(f"json['id']: {json.loads(car.model_dump_json(by_alias=True))['id']}")
assert json.loads(car_no_id.model_dump_json(by_alias=True))["id"] is None

# validate_assignment: atribuir string válida converte para UUID
car_no_id.id_ = "c4e60f4a-3c7f-4da5-9b3f-07aee50b23e7"
print(f"type(car_no_id.id_): {type(car_no_id.id_).__name__}")
print("testes extras: ok")

id='não-é-uuid': [(('id',), 'uuid_parsing')]
id=uuid1(): [(('id',), 'uuid_version')]
json['id']: c4e60f4a-3c7f-4da5-9b3f-07aee50b23e7
type(car_no_id.id_): UUID
testes extras: ok


## 4. Comparação com a solução do instrutor

| ponto | instrutor | esta solução |
|---|---|---|
| alias do `id_` | `Field(alias="id", default=None)` | nenhum `Field`: `make_alias` gera `id` |
| tipo | `UUID4 \| None` | `UUID4 \| None` |
| testes | os 2 do enunciado | + posição do campo, UUID inválido, UUID de outra versão, JSON, atribuição |

> ⚠️ **Ponto de design (antecipando a próxima seção):** `id = None` significa que um carro pode existir **sem identidade**. Isso é
> aceitável enquanto o dado não foi persistido, mas costuma virar bug (dois carros "iguais" sem id). A seção 6 resolve com
> `default_factory=uuid4`.